# GenAI Intensive 3일차 예제 노트북 · 벡터 DB · 문서 처리 파이프라인 · 첫 RAG

강의안의 3일차 예제 코드가 모두 이 노트북 하나에 있습니다. 셀 제목의 번호(예제 1-1 (a))는 강의안 코드 장 제목의 번호와 같습니다.
위에서부터 순서대로 실행합니다. 파트 1 은 벡터 DB, 파트 2 는 문서 처리 파이프라인, 파트 3 은 첫 RAG 입니다.
실습 미션은 따로 받은 `day03-lab.ipynb` 에서 합니다.

**처음 한 번만 할 일 · 코랩**
1. [파일] → [드라이브에 사본 저장] 을 선택하고, 새 탭에 열린 사본에서 진행합니다.
2. 왼쪽 열쇠 아이콘(보안 비밀)에 2일차에 넣은 `ANTHROPIC_API_KEY` 가 있는지 보고, 그 줄의 「노트북 액세스」를 켭니다. 없으면 「새 보안 비밀 추가」에서 이름 `ANTHROPIC_API_KEY`, 값에 받은 키를 붙여 넣습니다.
3. 아래 「환경 준비」 셀 세 개를 차례로 실행합니다(Shift + Enter). (1) 재료 · 라이브러리 → (2) 키 → (3) 연결 확인 순서입니다. (2) 셀에서 「데이터 접근」 허용 창이 뜨면 [허용]을 선택합니다. 마지막 셀에 `준비 완료` 가 나오면 끝입니다.

처음 실행할 때 임베딩 모델 `intfloat/multilingual-e5-small` 을 내려받습니다(약 470MB).

### Claude Code 에 물어보는 법

셀의 코드가 궁금하면 셀을 통째로 복사해 PC 의 Claude Code 에 붙이고, 아래 셋 가운데 하나를 덧붙여 묻습니다. Claude Code 가 알려 준 코드는 이 노트북의 새 셀에 붙여 실행합니다.

1. 「이 셀에서 ○○ 줄이 하는 일을 한 문장으로 알려 줘」
2. 「○○ 값을 △△ 로 바꿔 실행하면 결과가 어떻게 달라지는지 알려 줘」
3. 「이 오류가 왜 났고 어떻게 고치는지 알려 줘」(오류 메시지 전문을 함께 붙입니다)

셀마다 맨 끝 주석 `# Claude Code 에 물어보기:` 에 그 셀에 맞는 질문을 하나씩 적어 두었습니다.

## 환경 준비

### 환경 준비 (1) 실습 재료와 라이브러리

노트북을 열 때마다 가장 먼저 한 번 실행하는 셀입니다.

1. 코랩이면 학습 사이트에서 실습 자료 zip 두 개(`day03-examples.zip` · `day03-missions.zip`)를 내려받아 풉니다. 이미 파일 창에 올려 두었으면 그 파일을 씁니다.
2. 코랩에 없을 수 있는 라이브러리(`anthropic` · `sentence-transformers` · `python-docx`)와 PDF 를 텍스트로 바꾸는 `pdftotext`(poppler-utils)를 설치합니다.
3. 버전과 파일 개수를 찍어 준비가 끝났는지 보여 줍니다.

**볼 것** 마지막 줄에 「문서 253 건 · 원본 문서 255 건 · 미션 재료 3 개」가 찍히면 됩니다. 0 건이 나오거나 `FileNotFoundError` · 「풀리지 않아 지웠습니다」가 나오면 셀을 다시 실행합니다. 풀리지 않은 zip 은 이 셀이 지우므로 다시 실행하면 새로 내려받습니다.

In [ ]:
# 환경 준비: 노트북을 열면 이 셀부터 한 번 실행합니다
# 코랩에서는 실습 자료 zip 두 개를 내려받아 풀고, 없는 라이브러리만 설치합니다.
import os, sys, glob, subprocess

IN_COLAB = 'google.colab' in sys.modules
SITE = 'https://lecture-web-five.vercel.app/materials/'

if IN_COLAB:
    os.chdir('/content')
    for z in ['day03-examples.zip', 'day03-missions.zip']:
        if os.path.exists(z) and os.path.getsize(z) == 0:
            os.remove(z)                                # 지난번 내려받기가 실패해 남은 빈 파일
        if not os.path.exists(z):                       # 파일 창에 직접 올렸으면 그대로 씁니다
            if subprocess.run(['wget', '-q', SITE + z, '-O', z]).returncode != 0:
                if os.path.exists(z):
                    os.remove(z)
                raise SystemExit(f'{z} 를 내려받지 못했습니다. 학습 사이트에서 받아 '
                                 f'왼쪽 파일 창에 올린 뒤 이 셀을 다시 실행하세요.')
        if subprocess.run(['unzip', '-q', '-o', z]).returncode != 0:
            os.remove(z)                                # 풀리지 않는 파일은 지우고 다음 실행에서 다시 받습니다
            raise SystemExit(f'{z} 가 풀리지 않아 지웠습니다. 이 셀을 다시 실행하세요.')
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                    'anthropic', 'sentence-transformers', 'python-docx'])
    subprocess.run('apt-get -qq install -y poppler-utils > /dev/null', shell=True)

import numpy, sentence_transformers
print('numpy', numpy.__version__, '· sentence-transformers', sentence_transformers.__version__)
print('문서', len(os.listdir('corpus')), '건 · 원본 문서',
      len(glob.glob('docs_raw/*/*')), '건 · 미션 재료',
      len(glob.glob('3일차_미션*_재료.txt')), '개')

# Claude Code 에 물어보기: IN_COLAB 이 False 일 때 이 셀이 건너뛰는 일을 알려 줘

### 환경 준비 (2) API 키를 환경 변수로 옮기기

코랩 보안 비밀에 저장한 API 키를 읽어 환경 변수 `ANTHROPIC_API_KEY` 에 넣습니다. 이 셀을 건너뛰면 환경 준비 (3) 에서 `TypeError: Could not resolve authentication method` 오류가 납니다.
- 처음 실행하면 「데이터 접근」 허용 창이 뜹니다. [허용]을 선택합니다.

**볼 것** 키 앞부분이 `sk-ant-` 로 시작하는지 확인합니다. 키를 읽지 못하면 확인할 항목 세 가지가 출력됩니다. 차례로 확인한 뒤 셀을 다시 실행합니다.

In [ ]:
# 환경 준비 (2) 보안 비밀의 키를 환경 변수로 옮깁니다 · 키를 코드에 적지 않습니다
import os
from google.colab import userdata                 # 코랩 보안 비밀

try:
    os.environ['ANTHROPIC_API_KEY'] = userdata.get('ANTHROPIC_API_KEY').strip()
    print('키 앞부분', os.environ['ANTHROPIC_API_KEY'][:7],
          '· 길이', len(os.environ['ANTHROPIC_API_KEY']))
except Exception as e:
    print('보안 비밀에서 키를 읽지 못했습니다:', type(e).__name__)
    print('1. 열쇠 아이콘의 이름이 ANTHROPIC_API_KEY 와 같은지 봅니다')
    print('2. 그 줄의 「노트북 액세스」가 켜져 있는지 봅니다')
    print('3. 「데이터 접근」 창이 떴다면 [허용]을 선택합니다')
    print('확인한 뒤 이 셀을 다시 실행합니다')

# Claude Code 에 물어보기: userdata.get 이 None 을 주면 어느 줄에서 오류가 나는지 알려 줘

### 환경 준비 (3) 클라이언트를 만들고 연결 확인하기

API 를 호출할 클라이언트를 만들고, 경량 모델에 짧은 요청을 한 번 보내 키와 연결을 확인합니다.
- **client**: API 서버와 통신하는 객체입니다. 이후 호출은 `client.messages.create(...)` 형태로 실행합니다.
- **모델 등급**: 경량(haiku)은 빠르고 저렴합니다. 중형(sonnet)은 3일차에 답변을 생성하는 기본 모델입니다.

**볼 것** SDK 버전과 `준비 완료` 가 나오면 준비가 끝났습니다. `TypeError: Could not resolve authentication method` 가 나오면 키가 환경 변수에 없는 것이니 환경 준비 (2) 셀부터 다시 실행합니다. `AuthenticationError`(401)가 나오면 키 값이 틀렸거나 지운 키입니다. 보안 비밀에 넣은 키 값을 다시 확인합니다.

In [ ]:
# 환경 준비 (3) 클라이언트를 만들고 연결을 확인합니다
import anthropic

client = anthropic.Anthropic()        # 환경 변수 ANTHROPIC_API_KEY 를 읽습니다
MODEL       = 'claude-sonnet-5-5'     # 중형 · 3일차 답변 생성 모델(조립 셀과 같은 값)
SMALL_MODEL = 'claude-haiku-4-5'      # 경량 · 연결 확인용
print('anthropic', anthropic.__version__)

r = client.messages.create(model=SMALL_MODEL, max_tokens=20,
        messages=[{'role': 'user', 'content': '"준비 완료" 라고만 답해줘.'}])
print(''.join(b.text for b in r.content if b.type == 'text'))

# 이렇게 나오면 준비가 끝난 것입니다
#   anthropic (버전 숫자)
#   준비 완료

# Claude Code 에 물어보기: r.content 에 text 말고 어떤 type 의 조각이 올 수 있는지 알려 줘

## 파트 1 · 벡터 DB 와 RAG 구조

예제 1-1 · 1-2 입니다. 이 파트는 API 키가 필요 없습니다.

## 예제 1-1 · 벡터 DB 를 만들고 저장 · 검색 · 갱신하기

**목적** 2일차에 만든 임베딩 인덱스(문서 253건 × 384차원)를 다시 만들고, 2일차와 같은 질문 5개로 recall@k 를 다시 확인해 몇 건을 가져올지(k) 정한 뒤,
파일로 저장하고 다시 불러오고, 문서가 개정·추가·삭제됐을 때 인덱스의 그 행만 고쳐 검색 결과가 바뀌는 것을 확인합니다.

### (a) 2일차 인덱스를 다시 만들기

문서 253건을 벡터 253개로 바꿉니다.

**볼 것** 처음 실행하면 모델을 읽는 진행 막대(`Loading weights`)가 먼저 찍히고, 그 뒤에 「253 건 · 인덱스 (253, 384) · 0.389 MB」가 나옵니다. 253건 × 384차원 × 4바이트 = 0.389MB 입니다. 이 배열이 오늘의 벡터 DB 입니다.

**짚을 줄**
- `'passage: '` 는 e5 모델이 문서 앞에 붙이라고 정한 머리말입니다. 질문 앞에는 `'query: '` 를 붙입니다.
- `normalize_embeddings=True` 는 벡터 길이를 1 로 맞춥니다. 그래서 뒤에서 `E @ v`(행렬 곱) 한 번이 곧 코사인 유사도가 됩니다.
- `E.nbytes` 는 배열이 차지하는 바이트 수입니다. `1e6` 으로 나눠 MB 로 봅니다.

In [ ]:
# (a) 2일차 인덱스를 다시 만듭니다 (문서 253건 → 벡터 253개)
import os, json, unicodedata
import numpy as np
from sentence_transformers import SentenceTransformer

files = sorted(f for f in os.listdir('corpus') if f.endswith('.md'))
names = [unicodedata.normalize('NFC', f[:-3]) for f in files]     # 맥에서 푼 한글 파일명 대비
docs  = [open(f'corpus/{f}', encoding='utf-8').read() for f in files]

model = SentenceTransformer('intfloat/multilingual-e5-small')
E = model.encode(['passage: ' + d for d in docs],
                 normalize_embeddings=True, batch_size=32)
print(len(docs), '건 · 인덱스', E.shape, '·', round(E.nbytes / 1e6, 3), 'MB')

# Claude Code 에 물어보기: E 의 첫 행 길이(norm)를 찍어 1 인지 확인하는 코드 짜 줘

### (b) 2일차 질문 5개로 다시 재기

같은 질문 · 같은 정답 문서로 어제와 같은 순위가 나오는지 봅니다.

**볼 것** 질문 5개 중 정답이 1위인 것은 80% 입니다.

**짚을 줄**
- `(E @ v)` 는 문서 253건 각각과 질문의 유사도 253개입니다. `.argsort()[::-1]` 은 그 점수를 큰 순서로 늘어놓은 문서 번호 목록입니다.
- `r.index(names.index(gold)) + 1` 은 정답 문서의 행 번호가 그 목록의 몇 번째인지(0부터 셈) 찾고, 1 을 더해 「몇 위」로 바꿉니다.

In [ ]:
# (b) 질문 다섯 개로 정답 문서가 몇 위에 오는지 잽니다
QSET = [
    ("칠러가 안 도는 것 같은데 챔버가 멈추는 기준이 뭔가요", 'SOP-ETCH-007_v2.1'),
    ("에치 리크 체크는 어떻게 하나요",                      'SOP-ETCH-006'),
    ("CVD 알람이 뜨면 뭘 먼저 하나요",                      'SOP-CVD-009'),
    ("He 배면압 설정 기준이 어떻게 되나요",                  'SOP-ETCH-012_v2.0'),
    ("포토 웨이퍼를 수동으로 넣는 절차",                     'SOP-포토-013'),
]
ranks = []
for q, gold in QSET:
    v = model.encode(['query: ' + q], normalize_embeddings=True)[0]
    r = list((E @ v).argsort()[::-1])
    ranks.append(r.index(names.index(gold)) + 1)
    print(f'{gold:22s} {ranks[-1]}위')

# Claude Code 에 물어보기: 2위인 SOP-ETCH-012 질문에서 1위로 온 문서가 무엇인지 찍어 줘

### (c) recall@k 를 다시 내기

(심화 · 수업에서 넘어갈 수 있음) 넘어가도 뒤 셀은 그대로 돌아갑니다.

2일차와 같은 80% · 100% 가 나오면 인덱스를 제대로 다시 만든 것입니다.

**볼 것** k 를 3 으로 올리면 질문 5개의 정답이 전부 들어옵니다. 실습 미션 1 에서 상위 몇 건까지 볼지 정하는 `K = 3` 은 이 결과로 정한 값입니다.

In [ ]:
# (c) k 를 바꿔 가며 recall@k 를 냅니다 (정답이 상위 k건 안에 든 질문의 비율)
for k in [1, 3, 5, 10]:
    print(f'recall@{k:<2d}', round(sum(r <= k for r in ranks) / len(ranks) * 100), '%')

# Claude Code 에 물어보기: 정답이 2위 이하인 질문만 골라 찍는 코드로 고쳐 줘

### (d) 인덱스를 파일로 저장

매번 다시 임베딩하지 않습니다.

**볼 것** 「저장 완료 · 379 KB」가 찍히고, 왼쪽 파일 창에 `index.npy` · `index_names.json` 두 파일이 생겼는지 확인합니다.

In [ ]:
# (d) 저장: 인덱스(벡터)와 이름표(문서 이름)를 파일 두 개로 남깁니다
np.save('index.npy', E)
json.dump(names, open('index_names.json', 'w', encoding='utf-8'), ensure_ascii=False)
print('저장 완료 ·', os.path.getsize('index.npy') // 1024, 'KB')

# Claude Code 에 물어보기: index.npy 를 float16 으로 저장하면 크기가 얼마가 되는지 알려 줘

### (e) 저장한 인덱스 다시 불러오기

다음에 노트북을 열 때는 문서를 다시 임베딩하지 않고 이 셀로 인덱스를 불러옵니다.

**볼 것** 「(253, 384) 253」이 찍히는지 확인합니다. 새로 연 노트북이라면 예제 1-1 (a) 의 파일 읽기 · 모델 읽기 줄은 다시 실행합니다(예제 1-1 (f) 가 `model` · `docs` 를 씁니다). 다시 하지 않는 것은 문서 임베딩(`model.encode`) 줄입니다.

In [ ]:
# (e) 불러오기: 다음에 열 때는 임베딩을 다시 돌리지 않고 이 두 줄로 인덱스를 불러옵니다
E = np.load('index.npy')
names = json.load(open('index_names.json', encoding='utf-8'))
print(E.shape, len(names))

# Claude Code 에 물어보기: np.load 한 E 가 저장 전 E 와 똑같은지 비교하는 코드 짜 줘

### (f) 문서 한 건이 개정되면

전부 다시 만들지 않습니다.

**볼 것** 「다시 임베딩한 행 1개 · 인덱스 (253, 384)」가 찍힙니다. 행 수는 그대로이고 i 번째 행 하나만 새 벡터로 바뀌었습니다.

**짚을 줄**
- `rank_of` 는 예제 1-1 (b) 와 같은 방법으로 한 질문에서 문서 한 건이 몇 위인지 돌려주는 함수입니다.
- `docs[i] += …` 로 개정 내용을 본문 끝에 붙이고, `E[i] = …` 로 그 행 하나만 새 벡터로 덮어씁니다. 나머지 252행은 그대로입니다.

In [ ]:
# (f) 갱신: SOP 한 건이 개정되면 그 문서의 행 하나만 다시 임베딩합니다
Q_NEW = '온도 초과 판정을 몇 분으로 단축했나요'
i = names.index('SOP-ETCH-007_v2.1')

def rank_of(q, idx):
    v = model.encode(['query: ' + q], normalize_embeddings=True)[0]
    return list((E @ v).argsort()[::-1]).index(idx) + 1

before = rank_of(Q_NEW, i)
docs[i] += '\n\n## 2026-10 개정\n- 상한 초과 판정 시간을 5분에서 3분으로 단축합니다.'
E[i] = model.encode(['passage: ' + docs[i]], normalize_embeddings=True)[0]
np.save('index.npy', E)
print('다시 임베딩한 행 1개 · 인덱스', E.shape)

# Claude Code 에 물어보기: docs[i] 의 개정 내용을 corpus 파일에도 쓰려면 어떻게 할지 알려 줘

### (g) 갱신이 검색에 반영됐는지 보기

개정 내용을 묻는 질문으로 반영 전후 순위를 잽니다.

**볼 것** 행 하나만 다시 임베딩했는데 개정 내용을 묻는 질문의 순위가 바뀌었습니다. 갱신이 검색에 반영된 것입니다.

In [ ]:
# (g) 갱신이 검색에 반영됐는지 같은 질문으로 확인합니다
after = rank_of(Q_NEW, i)
print(f'개정 내용을 묻는 질문에서 SOP-ETCH-007 순위: {before}위 → {after}위')

# Claude Code 에 물어보기: Q_NEW 에서 2위·3위 문서 이름과 점수도 같이 찍어 줘

### (h) 문서 추가와 삭제

새 공지 1건을 붙이고 구판 SOP 1건을 뺍니다.

**볼 것** 행 수와 `names` 길이가 둘 다 253 인지(1건 추가 · 1건 삭제), 상위 3건의 첫자리에 새 공지 NOTI-099 가 왔는지 확인합니다.

**짚을 줄**
- `np.vstack` 은 배열 아래에 새 행을 붙이고, `np.delete(E, j, axis=0)` 은 j 번째 행을 지웁니다(`axis=0` 은 행 방향).
- 벡터를 붙이거나 지울 때마다 `names` · `docs` 도 같은 자리에서 함께 고칩니다. 하나라도 빠지면 행과 문서 이름이 어긋납니다.

In [ ]:
# (h) 추가와 삭제: 새 공지 1건을 붙이고, 구판 SOP-ETCH-012 v1.2 를 뺍니다
new_name = 'NOTI-099'
new_doc = ('# NOTI-099 사내 공지 — ESC 온도 기준 개정 안내\n'
           '- 게시일: 2026-10-01 · 게시 부서: 설비기술팀\n'
           'SOP-ETCH-007 의 상한 초과 판정 시간이 5분에서 3분으로 바뀌었습니다.')
E = np.vstack([E, model.encode(['passage: ' + new_doc], normalize_embeddings=True)])
names.append(new_name); docs.append(new_doc)

j = names.index('SOP-ETCH-012_v1.2')
E = np.delete(E, j, axis=0); del names[j]; del docs[j]

np.save('index.npy', E)
json.dump(names, open('index_names.json', 'w', encoding='utf-8'), ensure_ascii=False)
v = model.encode(['query: ' + Q_NEW], normalize_embeddings=True)[0]
print(E.shape, len(names), '· 상위 3건', [names[t] for t in (E @ v).argsort()[::-1][:3]])

# Claude Code 에 물어보기: E · names · docs 길이가 어긋나면 멈추는 assert 를 넣어 줘

## 예제 1-2 · 임베딩 모델을 바꿔 같은 질문으로 재기

(심화 · 수업에서 넘어갈 수 있음) 넘어가도 뒤 예제가 쓰는 질문 10개는 파트 2 준비 셀이 다시 만듭니다.

**목적** 같은 문서 253건과 질문 10개를 두 임베딩 모델(한국어를 포함해 학습한 e5-small, 영어 문장으로만 학습한 MiniLM)에 넣어
recall@1 · @3 를 비교합니다. 모델 카드의 설명보다 우리 문서로 잰 숫자로 모델을 고른다는 것을 확인합니다.

### (a) 질문 10개와 비교 함수

예제 1-1 의 5개에 에치 SOP 질문 5개를 더합니다.

**볼 것** 순위 목록의 여덟 번째 값이 8번(파티클 카운트) 질문의 순위로, e5-small 도 73위입니다. 정답 문서에는 「파티클 측정」 한 줄뿐이고, 파티클을 다룬 회의록 · 보고서가 더 가깝게 잡혔습니다.

**짚을 줄**
- `recall()` 은 모델 하나로 문서 253건과 질문 10개를 임베딩하고, 질문마다 정답 문서의 순위를 목록으로 돌려줍니다. 두 번째 값은 벡터 차원 수입니다.
- `rk` 줄의 순위 계산은 예제 1-1 (b) 와 같습니다. 점수가 큰 순서의 문서 번호 목록에서 정답 문서가 몇 번째인지 찾아 1 을 더합니다.
- `zip(QSET10, V)` 는 질문과 그 질문의 벡터를 짝지어 하나씩 꺼냅니다.
- `q_prefix` · `d_prefix` 는 e5 가 쓰는 `query: ` · `passage: ` 머리말 자리입니다. 머리말을 쓰지 않는 모델은 빈 문자열로 부릅니다.

In [ ]:
# (a) 질문 10개: 예제 1-1 의 5개에 에치 SOP 질문 5개를 더합니다
QSET10 = QSET + [
    ("시즈닝 런은 왜 도나요",              'SOP-ETCH-005'),
    ("ESC 교체할 때 주의할 점",            'SOP-ETCH-004'),
    ("파티클 카운트가 올라가면 어떻게 하나요", 'SOP-ETCH-005'),
    ("가스라인 퍼지 순서",                 'SOP-ETCH-003'),
    ("예방정비 주기가 어떻게 되나요",        'SOP-ETCH-002'),
]
docs0 = [open(f'corpus/{f}', encoding='utf-8').read() for f in files]   # 갱신 전 원본 253건
names0 = [unicodedata.normalize('NFC', f[:-3]) for f in files]

def recall(m, q_prefix='', d_prefix=''):
    D = m.encode([d_prefix + d for d in docs0], normalize_embeddings=True, batch_size=32)
    V = m.encode([q_prefix + q for q, _ in QSET10], normalize_embeddings=True)
    rk = [list((D @ v).argsort()[::-1]).index(names0.index(g)) + 1
          for (q, g), v in zip(QSET10, V)]
    return rk, D.shape[1]

rk_e5, dim = recall(model, 'query: ', 'passage: ')
print('e5-small   ', dim, '차원 · 순위', rk_e5)

# Claude Code 에 물어보기: 8번 질문에서 1~3위로 올라온 문서 이름을 찍어 줘

### (b) 영어 전용 모델로 다시 재기

처음 한 번 약 90MB 를 내려받습니다. 진행 막대(`Loading weights`) 뒤에 나오는 `MiniLM(영어)` 줄을 봅니다.

**볼 것** 차원은 같은데 정답이 상위 3건에 하나도 들지 않았습니다.

In [ ]:
# (b) 영어 전용 모델로 같은 질문 10개를 다시 잽니다 (약 90MB 를 처음 한 번 내려받습니다)
en = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')
rk_en, dim = recall(en)
print('MiniLM(영어)', dim, '차원 · 순위', rk_en)

# Claude Code 에 물어보기: MiniLM 에 'query: ' 접두를 붙이면 순위가 달라지는지 확인해 줘

### (c) 두 모델을 나란히 놓기

두 모델의 recall@1 · @3 과 최저 순위를 나란히 찍습니다.

**볼 것** recall@3 이 e5-small 80%, MiniLM 0% 입니다. 두 모델 모두 384차원이므로, 결과를 가른 것이 차원이 아니라 학습 언어인지 확인합니다.

In [ ]:
# (c) 두 모델을 recall@1 · @3 로 나란히 놓습니다
for label, rk in [('e5-small (다국어)', rk_e5), ('MiniLM (영어 전용)', rk_en)]:
    r1 = sum(r <= 1 for r in rk) * 10
    r3 = sum(r <= 3 for r in rk) * 10
    print(f'{label:18s} recall@1 {r1:3d}%  recall@3 {r3:3d}%  최저 순위 {max(rk)}위')

# Claude Code 에 물어보기: recall@5 와 평균 순위 열을 표에 추가해 줘

## 파트 2 · 문서 처리 파이프라인

예제 2-1 ~ 2-5 입니다. 이 파트는 API 키가 필요 없습니다. 첫 셀(준비)이 문서 253건을 다시 읽으므로, 환경 준비 셀 다음에 파트 1 을 건너뛰고 여기서 시작해도 됩니다.
`pdftotext` 는 환경 준비 (1) 셀에서 코랩에 설치됩니다.

### 준비 셀: 파트 1 의 값 다시 만들기 (문서 253건 · 임베딩 모델 · 질문 10개)

예제 1-1 (f) ~ (h) 에서 인덱스와 문서 목록을 고쳤으므로, 파트 2 는 이 셀에서 문서 253건 · 임베딩 모델 · 질문 10개를 처음 상태로 다시 만들고 시작합니다. 코드는 예제 1-1 (a) · 1-2 (a) 와 같습니다.

**볼 것** 진행 막대 뒤에 「253 건 · 질문 10 개」가 찍히면 다음 셀로 넘어갑니다.

In [ ]:
# 준비: 문서 253건 · 임베딩 모델 · 질문 10개 (예제 1-1 · 1-2 와 같은 값)
import os, re, json, unicodedata
import numpy as np
from sentence_transformers import SentenceTransformer

files = sorted(f for f in os.listdir('corpus') if f.endswith('.md'))
names = [unicodedata.normalize('NFC', f[:-3]) for f in files]
docs  = [open(f'corpus/{f}', encoding='utf-8').read() for f in files]
model = SentenceTransformer('intfloat/multilingual-e5-small')

QSET10 = [
    ("칠러가 안 도는 것 같은데 챔버가 멈추는 기준이 뭔가요", 'SOP-ETCH-007_v2.1'),
    ("에치 리크 체크는 어떻게 하나요",                      'SOP-ETCH-006'),
    ("CVD 알람이 뜨면 뭘 먼저 하나요",                      'SOP-CVD-009'),
    ("He 배면압 설정 기준이 어떻게 되나요",                  'SOP-ETCH-012_v2.0'),
    ("포토 웨이퍼를 수동으로 넣는 절차",                     'SOP-포토-013'),
    ("시즈닝 런은 왜 도나요",                              'SOP-ETCH-005'),
    ("ESC 교체할 때 주의할 점",                            'SOP-ETCH-004'),
    ("파티클 카운트가 올라가면 어떻게 하나요",                 'SOP-ETCH-005'),
    ("가스라인 퍼지 순서",                                 'SOP-ETCH-003'),
    ("예방정비 주기가 어떻게 되나요",                        'SOP-ETCH-002'),
]
print(len(docs), '건 · 질문', len(QSET10), '개')

# Claude Code 에 물어보기: QSET10 을 JSON 파일로 저장하고 읽는 두 줄 짜 줘

## 예제 2-1 · 배포 폴더를 열어 문서 확인하기

**목적** PDF·워드·엑셀·md 원본 255건이 든 `docs_raw/` 를 열어 형식별 건수를 세고, PDF 한 건을 텍스트로 뽑아
본문에 섞여 들어온 머리말·바닥글 줄이 몇 %인지 숫자로 확인합니다.

### (a) 배포 폴더 훑기

앞에서 쓴 md 253건의 원본 파일을 형식별로 셉니다.

**볼 것** pdf 8 · docx 10 · xlsx 2 · md 235, 합 255건이 찍힙니다. 표인 엑셀 2개를 빼면 앞에서 쓴 253건과 같은 수인지 확인합니다.

In [ ]:
# (a) 배포 폴더를 형식별로 셉니다
for k in ['pdf', 'docx', 'xlsx', 'md']:
    files_k = [f for f in os.listdir(f'docs_raw/{k}') if not f.startswith('.')]
    print(k, len(files_k))

# Claude Code 에 물어보기: 형식별 건수 옆에 파일 크기 합계(MB)도 같이 찍어 줘

### (b) PDF 한 건을 텍스트로

표가 든 PDF 한 건(SOP-ETCH-007)부터 텍스트로 뽑습니다.

**볼 것** 「953 자」가 찍힙니다. corpus 의 정제본은 603자였으니 350자가 더 나온 것입니다.

In [ ]:
# (b) PDF 한 건을 텍스트로 뽑습니다
import subprocess

PDF = 'docs_raw/pdf/SOP-ETCH-007_v2.1.pdf'
txt = subprocess.run(['pdftotext', PDF, '-'],
                     capture_output=True, text=True).stdout
print(len(txt), '자')

# Claude Code 에 물어보기: pdftotext 가 없는 환경에서 쓸 파이썬 대안을 알려 줘

### (c) 노이즈가 몇 %인지 재기

숫자로 확인합니다.

**볼 것** 「4 / 57 줄 · 7.0 %」가 찍힙니다. 이 문서는 한 쪽짜리입니다. 머리말 · 바닥글 · 쪽번호는 쪽마다 되풀이되므로, 쪽이 많은 문서에서는 지울 줄도 늘어납니다.

**짚을 줄**
- `any(k in l for k in DROP)` 는 DROP 낱말 가운데 하나라도 그 줄에 들어 있으면 참입니다. 줄의 일부만 맞아도 걸리므로, 본문 줄에 같은 낱말이 있으면 그 줄도 노이즈로 셉니다.
- `re.fullmatch(r'\s*-\s*\d+\s*-\s*', l)` 은 줄 전체가 「- 3 -」 같은 쪽번호 모양인지 봅니다(`\d+` 는 숫자, `\s*` 는 빈칸 0개 이상).

In [ ]:
# (c) 노이즈 줄이 몇 %인지 잽니다
DROP = ['무단 반출 금지', '문서관리시스템', '문서번호']
lines = [l for l in txt.split('\n') if l.strip()]
noise = [l for l in lines
         if any(k in l for k in DROP) or re.fullmatch(r'\s*-\s*\d+\s*-\s*', l)]
print(len(noise), '/', len(lines), '줄 ·', round(len(noise) / len(lines) * 100, 1), '%')

# Claude Code 에 물어보기: DROP 규칙이 본문 줄을 잘못 잡았는지 noise 를 찍어 확인해 줘

## 예제 2-2 · 형식마다 여는 법이 다릅니다

**목적** PDF(표 포함)·DOCX·XLSX 를 각각 맞는 도구로 열어, 노이즈를 지우고 제목 구조와 표를 살려 뽑습니다.
표를 통째로 넣으면 모델 입력 한계 뒤의 행이 벡터에 빠진다는 것을 재고, 한 행을 한 문장으로 풀어 해결합니다.

### (a) 노이즈 제거 규칙 만들기

반복되는 줄을 제거합니다.

**볼 것** 「101 줄 → 97 줄」이 찍힙니다. 빠진 4줄이 예제 2-1 (c) 에서 센 노이즈 4줄입니다.

**짚을 줄**
- 예제 2-1 (c) 와 같은 두 조건을 `not` 으로 뒤집어, 노이즈가 아닌 줄만 남깁니다. 여기서는 빈 줄도 세므로 예제 2-1 (c) 의 57줄과 줄 수가 다릅니다.

In [ ]:
# (a) 노이즈 제거 규칙을 적용합니다
keep = [l for l in txt.split('\n')
        if not any(k in l for k in DROP)
        and not re.fullmatch(r'\s*-\s*\d+\s*-\s*', l)]
print(len(txt.split('\n')), '줄 →', len(keep), '줄')

# Claude Code 에 물어보기: keep 에서 빈 줄까지 빼면 몇 줄이 남는지 찍어 줘

### (b) 표가 든 PDF 를 옵션 없이 뽑으면

(심화 · 수업에서 넘어갈 수 있음) 넘어가도 뒤 셀은 그대로 돌아갑니다.

같은 파일의 부표 구간입니다.

**볼 것** 한 셀이 한 줄이 됐습니다. 어느 값이 어느 열인지 알 수 없습니다.

In [ ]:
# (b) 표가 든 PDF 를 옵션 없이 뽑으면 (같은 파일의 부표 구간)
print(txt[-200:])

# Claude Code 에 물어보기: txt 에서 부표 구간이 몇 줄로 흩어졌는지 세어 줘

### (c) -layout 을 주면

열 위치를 살려서 뽑습니다.

**볼 것** 옵션 없이 뽑으면 칸마다 한 줄씩 흩어지던 부표(예제 2-2 (b))가 한 행 한 줄로 남습니다. 칸 사이는 공백뿐이라, 표를 검색용 문장으로 푸는 법은 예제 2-2 (h) 에서 봅니다.

**짚을 줄**
- `rows[-9:-1]` 은 빈 줄을 뺀 줄 가운데 끝에서 9번째부터 마지막 줄 바로 앞까지, 곧 부표 8줄(제목 · 열 이름 · 호기 6행)입니다. 마지막 줄은 바닥글(최신본 안내와 쪽번호 「-1-」)입니다.

In [ ]:
# (c) -layout 을 주면 열 위치를 살려 뽑습니다 ((b) 와 같은 부표 구간)
lay = subprocess.run(['pdftotext', '-layout', PDF, '-'],
                     capture_output=True, text=True).stdout
rows = [l for l in lay.split('\n') if l.strip()]
print('\n'.join(rows[-9:-1]))

# Claude Code 에 물어보기: rows 의 부표 8줄을 공백 기준으로 다섯 칸으로 나눠 줘

### (d) 워드 파일 열기

python-docx 는 문단과 스타일을 그대로 줍니다.

**볼 것** 「11 문단」이 찍힙니다. 빈 문단은 세지 않습니다. 제목 구조는 예제 2-2 (e) 에서 뽑습니다.

In [ ]:
# (d) 워드 파일을 엽니다
from docx import Document

doc = Document('docs_raw/docx/MTG-2025-W01.docx')
print(len([p for p in doc.paragraphs if p.text.strip()]), '문단')

# Claude Code 에 물어보기: 이 docx 안에 표(doc.tables)가 있는지 세어 줘

### (e) 스타일로 구조 뽑기

(심화 · 수업에서 넘어갈 수 있음) 넘어가도 뒤 셀은 그대로 돌아갑니다.

제목인지 본문인지 파일이 알려줍니다.

**볼 것** Heading 1 한 줄과 Heading 2 두 줄이 찍힙니다. 이 제목으로 절 단위로 자르는 방법은 예제 2-3 (e) 에서 봅니다.

In [ ]:
# (e) 스타일로 제목 구조를 뽑습니다
for p in doc.paragraphs:
    if p.style.name.startswith('Heading'):
        print(p.style.name, p.text)

# Claude Code 에 물어보기: Heading 2 아래 본문 문단을 절 이름과 묶어 출력해 줘

### (f) 엑셀 열기

첫 줄이 열 이름이 아닙니다.

**볼 것** 1행이 병합된 제목입니다. 그대로 읽으면 컬럼명이 안 나옵니다.

In [ ]:
# (f) 엑셀을 엽니다: 첫 줄이 열 이름이 아닙니다
from openpyxl import load_workbook

XL = 'docs_raw/xlsx/공정스펙_관리한계_일람.xlsx'
ws = load_workbook(XL).active
print([c.value for c in ws[1]][:3])

# Claude Code 에 물어보기: 2행과 3행에 무엇이 들어 있는지 찍어 줘

### (g) 장식 줄을 건너뛰고 다시

장식 줄 세 줄을 건너뛰고 읽습니다.

**볼 것** 열 이름 「문서번호 · 문서명 · 적용 장비 · 개정일」이 찍힙니다. 예제 2-2 (f) 에서 첫 줄이 열 이름이 아니었던 것은 위 3줄이 제목 · 출력일 · 빈 줄이기 때문입니다.

In [ ]:
# (g) 장식 줄 세 줄을 건너뛰고 다시 읽습니다
import pandas as pd

df = pd.read_excel(XL, skiprows=3)
print(list(df.columns)[:4])

# Claude Code 에 물어보기: skiprows 를 2 로 하면 열 이름이 어떻게 되는지 보여 줘

### (h) 표 한 행을 한 문장으로

30행짜리 스펙 표를 행마다 한 문장으로 풀어 씁니다.

**볼 것** 표를 통째로 넣으면 1,797자짜리 청크 하나입니다. 풀면 81자짜리 30개가 됩니다.

**짚을 줄**
- `df.iterrows()` 는 표를 한 행씩 꺼냅니다. `r['문서번호']` 처럼 열 이름으로 칸 값을 읽어 f-문자열 한 문장에 끼웁니다.
- `sum(map(len, sent)) / len(sent)` 는 문장 길이의 평균입니다.

In [ ]:
# (h) 표 한 행을 한 문장으로 풉니다
sent = [f"{r['문서번호']} 은 {r['적용 장비']} 호기에 적용되는 {r['문서명']} 이고, "
        f"개정일은 {r['개정일']} · 버전은 {r['버전']} 입니다."
        for _, r in df.iterrows()]
print(len(sent), '문장 · 평균', round(sum(map(len, sent)) / len(sent)), '자')
print(sent[1])

# Claude Code 에 물어보기: 적용 장비 칸이 비어 있는 행이 있으면 문장이 어떻게 되는지 확인해 줘

### (i) 표를 통째로 넣으면 뒤쪽 행이 빠집니다

모델 입력 한계 512 토큰 뒤의 행은 벡터에 들어가지 않습니다.

**볼 것** 첫 줄의 「(722 > 512)」 경고는 글이 모델 한계보다 길다는 알림이고 오류가 아닙니다. 마지막 8행을 지운 표와의 유사도가 1.0 이면 그 8행은 벡터에 아무 영향이 없었다는 뜻입니다. 마지막 줄에서 SPEC-030 질문의 1위가 SPEC-030 행 문장인지 확인합니다.

**짚을 줄**
- `model.tokenizer(...)['input_ids']` 의 길이는 글 전체의 토큰 수(722)입니다. 모델은 앞의 `model.max_seq_length`(512) 토큰만 읽고, 넘는 뒷부분은 잘려 벡터에 들어가지 않습니다.
- `df.iloc[:-8]` 은 마지막 8행을 뺀 표입니다. 두 벡터의 유사도가 1.0 이면 뒤 8행이 벡터에 아무 영향을 주지 못했다는 뜻입니다.
- `pool = docs + sent` 의 `docs` 는 파트 2 준비 셀에서 읽은 문서 253건, `sent` 는 예제 2-2 (h) 의 행 문장 30개입니다.
- `(P @ v).argmax()` 는 유사도가 가장 큰 청크의 번호입니다.

In [ ]:
# (i) 표를 통째로 한 청크로 넣으면: 모델 입력 한계(512 토큰) 뒤의 행은 벡터에 들어가지 않습니다
table_blob = df.to_string(index=False)
print('표 전체', len(model.tokenizer('passage: ' + table_blob)['input_ids']), '토큰 · 모델 한계',
      model.max_seq_length, '토큰')
cut = df.iloc[:-8].to_string(index=False)                 # 마지막 8행(SPEC-023 ~ 030)을 지운 표
a, b = model.encode(['passage: ' + table_blob, 'passage: ' + cut], normalize_embeddings=True)
print('마지막 8행을 지운 표와의 유사도', round(float(a @ b), 4))

# 행마다 한 문장으로 풀면 30행이 각자 벡터를 갖습니다
q = 'SPEC-030 는 언제 개정됐나요?'
pool = docs + sent
P = model.encode(['passage: ' + c for c in pool], normalize_embeddings=True, batch_size=64)
v = model.encode(['query: ' + q], normalize_embeddings=True)[0]
print(q, '→ 1위:', pool[(P @ v).argmax()][:46])

# Claude Code 에 물어보기: 표를 10행씩 세 덩어리로 나눠 각각 토큰 수를 찍어 줘

## 예제 2-3 · 청크를 자르는 방법

**목적** 같은 문서를 글자 수 · 문장 경계 · 중첩 · 문서 구조(## 절) 네 가지로 잘라, 잘린 자리와 청크 개수가 어떻게 달라지는지 봅니다.

### (a) 가장 단순한 방법: 글자 수로 자르기

300자씩 끊습니다.

**볼 것** 「1 조각」이 찍힙니다. 첫 문서(`docs[0]`)가 300자보다 짧아 한 조각입니다. 글자 수만 보고 자르므로, 300자를 넘는 문서는 문장 한가운데서도 잘립니다. 잘린 자리는 예제 2-3 (b) 에서 봅니다.

In [ ]:
# (a) 가장 단순한 방법: 글자 수로 자르기
def chunk_fixed(text, size=300, overlap=0):
    step = size - overlap
    return [text[i:i + size]
            for i in range(0, len(text), step)
            if text[i:i + size].strip()]

print(len(chunk_fixed(docs[0])), '조각')

# Claude Code 에 물어보기: overlap 을 size 보다 크게 주면 무슨 일이 생기는지 알려 줘

### (b) 잘린 지점을 눈으로 보기

경계에서 무슨 일이 생기는지 확인합니다.

**볼 것** 문장 한가운데서 끊겨 두 조각 어느 쪽도 온전한 뜻을 갖지 못합니다.

**짚을 줄**
- `repr()` 로 찍으면 줄바꿈이 `\n` 으로 보여, 청크가 어디서 잘렸는지 눈으로 확인하기 쉽습니다.

In [ ]:
# (b) 잘린 자리를 눈으로 봅니다
i = names.index('SOP-ETCH-007_v2.1')
cs = chunk_fixed(docs[i], 100)
print(repr(cs[1][-30:]))
print(repr(cs[2][:30]))

# Claude Code 에 물어보기: cs 의 모든 경계 앞뒤 10자를 한 줄씩 찍어 줘

### (c) 문장 경계로 잘라 보기

마침표에서 끊고 길이를 맞춥니다.

**볼 것** 「3 조각 · [275, 296, 27]」이 찍힙니다. 한국어 문장은 대부분 「다」로 끝나서, 마침표가 없는 줄도 「다」 뒤 공백에서 나뉩니다.

**짚을 줄**
- `re.split(r'(?<=[.!?다])\s+', text)` 는 「. ! ? 다」 바로 뒤의 빈칸에서 문장을 나눕니다. `(?<=…)` 는 그 글자를 지우지 않고 바로 뒤에서 자르라는 표시입니다.
- 한계: 「그러다 보니」처럼 「다」로 끝나는 낱말 뒤와 줄바꿈에서도 잘리므로 완벽한 문장 분리는 아닙니다. 정확히 하려면 문장 분리 라이브러리(예: kss)를 씁니다.
- 반복문은 문장을 `cur` 에 이어 붙이다가 `target`(300자)을 넘기기 직전에 한 조각으로 내보냅니다. 문장 하나가 300자보다 길면 자르지 않고 그대로 한 조각이 되므로, 300자를 넘는 조각이 나올 수 있습니다.

In [ ]:
# (c) 문장 경계로 잘라 봅니다
def chunk_sent(text, target=300):
    sents = re.split(r'(?<=[.!?다])\s+', text)
    out, cur = [], ''
    for s in sents:
        if cur and len(cur) + len(s) > target:
            out.append(cur.strip()); cur = ''
        cur += s + ' '
    if cur.strip():
        out.append(cur.strip())
    return out

print(len(chunk_sent(docs[i])), '조각 ·', [len(c) for c in chunk_sent(docs[i])])

# Claude Code 에 물어보기: '그러다 보니' 처럼 잘못 잘리는 자리가 docs[i] 에 있는지 찾아 줘

### (d) 중첩을 줘 보기

(심화 · 수업에서 넘어갈 수 있음) 넘어가도 뒤 셀은 그대로 돌아갑니다.

청크 수가 얼마나 늘어나는지 봅니다.

**볼 것** 「282 351 1.24」가 찍힙니다. 중첩 80자를 주면 청크가 1.24배가 되고, 임베딩할 양과 인덱스 크기도 그만큼 늘어납니다.

In [ ]:
# (d) 중첩을 줘 봅니다: 청크 수가 얼마나 늘어나는지
a = sum(len(chunk_fixed(t, 400, 0))  for t in docs)
b = sum(len(chunk_fixed(t, 400, 80)) for t in docs)
print(a, b, round(b / a, 2))

# Claude Code 에 물어보기: 청크 수 대신 전체 글자 수가 몇 배 늘었는지도 찍어 줘

### (e) 문서 구조로 자르기

`## ` 제목 단위로 끊고 문서 제목을 앞에 붙입니다.

**볼 것** 조각만 봐도 어느 문서의 어느 절인지 알 수 있게 만듭니다.

**짚을 줄**
- 첫 줄(`# 제목`)에서 문서 제목을 꺼내고, `## ` 로 시작하는 줄을 만날 때마다 그때까지 모은 줄을 (절 제목, 본문) 한 쌍으로 저장합니다.
- 첫 `## ` 앞의 머리 부분(제목 줄 · 적용 장비 · 개정일)은 절 제목 없이 첫 조각이 됩니다. 출력은 둘째 조각(`[1]`)입니다.
- 마지막 `return` 줄은 조각마다 `[문서 제목] 절 제목` 을 앞에 붙여 돌려줍니다. 본문이 빈 절은 버립니다.

In [ ]:
# (e) 문서 구조로 자르기: ## 절 단위로 끊고 앞에 [문서 제목] 을 붙입니다
def chunk_struct(text):
    title = text.split('\n', 1)[0].lstrip('# ').strip()
    parts, cur, head = [], [], None
    for ln in text.split('\n'):
        if ln.startswith('## '):
            if cur:
                parts.append((head, '\n'.join(cur)))
            head, cur = ln[3:].strip(), []
        else:
            cur.append(ln)
    if cur:
        parts.append((head, '\n'.join(cur)))
    return [f'[{title}] {h or ""}\n{b}'.strip() for h, b in parts if b.strip()]

print(chunk_struct(docs[i])[1][:80])

# Claude Code 에 물어보기: ## 이 하나도 없는 문서가 몇 건인지 세어 줘

### (f) 크기 셋과 절 단위로 잘라 개수 비교

(심화 · 수업에서 넘어갈 수 있음) 넘어가도 뒤 셀은 그대로 돌아갑니다.

같은 253건을 다르게 자릅니다.

**볼 것** 통문서로 두면 파트 2 준비 셀의 253개입니다. 100 · 200 · 400자 줄에서 잘게 자를수록 개수가 늘고, 절 단위는 757개인지 확인합니다.

In [ ]:
# (f) 크기 셋(100 · 200 · 400자)과 절 단위로 잘라 개수를 비교합니다
for size in [100, 200, 400]:
    print(size, sum(len(chunk_fixed(t, size)) for t in docs))
print('구조(## 단위)', sum(len(chunk_struct(t)) for t in docs))

# Claude Code 에 물어보기: 절 단위 조각의 평균 글자 수와 최대 글자 수를 찍어 줘

## 예제 2-4 · 청크가 검색 품질을 정말 바꾸는지 재기

**목적** 예제 2-3 의 자르기 방식마다 인덱스를 새로 만들고, 질문 10개의 정답 문서가 top-1 · top-3 안에 드는 비율을 재서
이 문서 묶음에 맞는 청크 크기를 숫자로 고릅니다.

### (a) 질문 10개로 적중률 재기

top-3 안에 정답 문서가 오는지 셉니다. 자르는 방식마다 청크를 새로 임베딩하므로 다른 셀보다 오래 걸립니다.

**볼 것** 다섯 줄의 top1 · top3 를 비교합니다. top3 는 400자와 통문서가 80% 로 가장 높습니다. 질문이 10개라 한 문항이 10% 입니다. 한두 개로 재면 우연과 구별되지 않습니다.

**짚을 줄**
- `QSET10` · `names` · `docs` 는 파트 2 준비 셀에서 만든 값입니다.
- `build_chunks(fn)` 은 자르기 함수 `fn` 으로 문서 253건을 자르고, 청크마다 어느 문서에서 나왔는지(`owner`)를 같이 적습니다.
- `evaluate` 는 청크를 임베딩한 뒤 질문마다 상위 3개 청크의 출처 문서를 봅니다. `h1 += tops[0] == gold` 는 참이면 1 이 더해지는 셈법입니다. `tops` 는 상위 청크 3개의 출처 문서라 같은 문서가 두 번 이상 들어올 수 있습니다.
- 마지막 반복문은 자르는 방식 다섯 가지를 차례로 넣어 같은 질문 10개로 잽니다. `lambda t: chunk_fixed(t, 100)` 은 「100자로 자르기」를 함수 하나로 넘기는 표기입니다.

In [ ]:
# (a) 자르기 방식마다 질문 10개로 적중률을 잽니다
QE = model.encode(['query: ' + q for q, _ in QSET10], normalize_embeddings=True)

def build_chunks(fn):
    chunks, owner = [], []
    for nm, t in zip(names, docs):
        for c in fn(t):
            chunks.append(c); owner.append(nm)
    return chunks, owner

def evaluate(chunks, owner):
    CE = model.encode(['passage: ' + c for c in chunks],
                      normalize_embeddings=True, batch_size=64)
    h1 = h3 = 0
    for k, (q, gold) in enumerate(QSET10):
        order = (CE @ QE[k]).argsort()[::-1]
        tops = [owner[j] for j in order[:3]]
        h1 += tops[0] == gold
        h3 += gold in tops
    n = len(QSET10)
    return len(chunks), round(h1 / n * 100), round(h3 / n * 100)

for label, fn in [('100자', lambda t: chunk_fixed(t, 100)),
                  ('200자', lambda t: chunk_fixed(t, 200)),
                  ('400자', lambda t: chunk_fixed(t, 400)),
                  ('통문서', lambda t: [t]),
                  ('구조(## 단위)', chunk_struct)]:
    n, t1, t3 = evaluate(*build_chunks(fn))
    print(f'{label:14s} 청크 {n:4d}  top1 {t1:3d}%  top3 {t3:3d}%')

# Claude Code 에 물어보기: top3 에 같은 문서가 중복으로 든 경우를 하나로 세게 고쳐 줘

### (b) 중첩을 준 뒤 적중률까지 재기

(심화 · 수업에서 넘어갈 수 있음) 넘어가도 뒤 셀은 그대로 돌아갑니다.

청크 수만 보지 말고 같은 질문 세트로 적중률을 함께 봅니다.

**볼 것** 청크가 469 → 598 로 28% 늘어도 top1 은 60% 그대로이고, top3 는 중첩 60자에서만 80% 로 오릅니다.

In [ ]:
# (b) 중첩을 0 · 20 · 40 · 60자로 바꿔 적중률까지 잽니다 (200자 기준)
for ov in [0, 20, 40, 60]:
    n, t1, t3 = evaluate(*build_chunks(lambda t: chunk_fixed(t, 200, ov)))
    print(f'중첩 {ov:2d}자  청크 {n:4d}  top1 {t1}%  top3 {t3}%')

# Claude Code 에 물어보기: 중첩 60자에서 top3 가 오른 질문이 어느 것인지 찾아 줘

## 예제 2-5 · 청크에 메타데이터(꼬리표) 붙이기

**목적** 청크마다 문서 유형 · 날짜 · 버전 · 부서 꼬리표를 붙이고, 그 꼬리표로 검색 범위를 좁혀
「회의록은 빼고」 「품질보증팀 문서 중 2026년 것만」 같은 조건을 유사도와 따로 처리할 수 있는지 확인합니다.

### (a) 문서 목록 파일에서 꼬리표 읽기

문서마다 유형 · 날짜 · 버전을 적어 둔 목록 파일(corpus_manifest.csv)을 읽습니다.

**볼 것** 출력 사전에 `doc_id` · `version` · `date` 가 있는지 확인합니다. 이 셋이 있어야 답에 출처를 달 수 있습니다.

**짚을 줄**
- `csv.DictReader` 는 CSV 한 줄을 {열 이름: 값} 사전으로 읽습니다. `encoding='utf-8-sig'` 는 엑셀로 저장한 CSV 맨 앞의 보이지 않는 표시(BOM)를 지우고 읽습니다.
- `META` 는 파일 이름을 열쇠로 꼬리표를 바로 찾는 사전입니다.

In [ ]:
# (a) 문서 목록 파일에서 꼬리표를 읽습니다
import csv

META = {unicodedata.normalize('NFC', r['file']): r
        for r in csv.DictReader(open('corpus_manifest.csv', encoding='utf-8-sig'))}
print(META['SOP-ETCH-012_v2.0.md'])

# Claude Code 에 물어보기: META 에 없는 corpus 파일이 있는지 확인해 줘

### (b) 필터를 걸어 검색 범위 좁히기

유형 꼬리표로 회의록 청크를 빼고, 검색 대상이 얼마나 줄어드는지 셉니다. 이 필터로 실제 검색하는 것은 예제 3-3 (c) 에서 봅니다.

**볼 것** 「청크 757 → 613 (19.0% 감소)」가 찍힙니다. 「기준이 뭔가」를 묻는 질문에서는 회의록이 노이즈가 될 수 있어 이렇게 뺍니다.

**짚을 줄**
- `build_chunks` 는 예제 2-4 (a), `chunk_struct` 는 예제 2-3 (e) 에서 만든 함수입니다. `owner` 는 청크마다 어느 문서에서 나왔는지 적은 목록입니다.

In [ ]:
# (b) 유형 꼬리표로 회의록을 걸러 검색 범위를 좁힙니다
chunks, owner = build_chunks(chunk_struct)
meta = [META[f'{o}.md'] for o in owner]
keep = [j for j, m in enumerate(meta) if m['type'] != '회의록']
print(f'청크 {len(chunks)} → {len(keep)} '
      f'({round((1 - len(keep) / len(chunks)) * 100, 1)}% 감소)')

# Claude Code 에 물어보기: 유형별 청크 수를 많은 순서로 찍어 줘

### (c) 꼬리표를 파일명에서 뽑기

(심화 · 수업에서 넘어갈 수 있음) 넘어가도 뒤 셀은 그대로 돌아갑니다.

파일 이름이 「유형-번호_v버전」 규칙을 따르면 꼬리표 일부를 이름에서 뽑을 수 있습니다.

**볼 것** 출력의 type · doc_id · version 이 예제 2-5 (a) 에서 읽은 META 값과 같은지 확인합니다.

**짚을 줄**
- 정규식은 「유형-번호_v버전」 꼴을 세 칸으로 나눕니다. `(SOP|QIR|SPEC|MTG)` 는 유형, `([\w가-힣-]+?)` 는 번호, `(?:_v([\d.]+))?` 는 있어도 없어도 되는 버전 칸입니다.
- `m[1]` · `m[2]` · `m[3]` 은 괄호 순서대로 잡힌 값입니다. 버전이 없으면 v1.0 으로 둡니다.

In [ ]:
# (c) 꼬리표를 파일명에서 뽑습니다
m = re.match(r'(SOP|QIR|SPEC|MTG)-([\w가-힣-]+?)(?:_v([\d.]+))?$',
             'SOP-ETCH-012_v2.0')
print({'type': m[1], 'doc_id': f'{m[1]}-{m[2]}',
       'version': f'v{m[3]}' if m[3] else 'v1.0'})

# Claude Code 에 물어보기: 'SOP-포토-013' 에 이 정규식을 적용한 결과를 보여 줘

### (d) 부서 · 날짜 꼬리표로 거르기

문서 머리말의 「작성 · 관리 부서 · 게시 부서」 줄에서 부서를 뽑습니다.

**볼 것** 첫 줄 맨 앞 `('-', 73)` 이 부서 꼬리표를 못 뽑은 문서 73건입니다. 회의록 48건과 매뉴얼 25건이 머리말에 부서 줄이 없는 문서입니다.

**짚을 줄**
- `dept_of` 의 정규식은 「작성 · 관리 부서 · 게시 부서」 가운데 하나 뒤에 오는 낱말(빈칸이나 · 앞까지)을 부서로 잡습니다.
- `collections.Counter(...).most_common(6)` 은 부서별 문서 수를 많은 순서로 여섯 개 보여 줍니다.
- 날짜 비교 `>= '2026-01-01'` 은 「연-월-일」 글자 순서가 날짜 순서와 같아서 문자열끼리 비교해도 됩니다.

In [ ]:
# (d) 부서 꼬리표를 문서 머리말에서 뽑고, 부서 · 날짜로 거릅니다
def dept_of(text):
    m = re.search(r'(?:작성|관리 부서|게시 부서): *([^\s·]+)', text)
    return m[1] if m else '-'

DEPT = {nm: dept_of(t) for nm, t in zip(names, docs)}
import collections
print(collections.Counter(DEPT.values()).most_common(6))

sel = [j for j, o in enumerate(owner)
       if DEPT[o] == '품질보증팀' and META[f'{o}.md']['date'] >= '2026-01-01']
print('품질보증팀 · 2026년 문서의 청크', len(sel), '/', len(chunks))

# Claude Code 에 물어보기: 부서를 못 뽑은 73건의 유형별 건수를 세어 줘

## 파트 3 · 첫 RAG 완성

예제 3-1 ~ 3-4 입니다. 답변 생성은 Claude API 를 직접 호출하므로 환경 준비 (2) · (3) 셀로 키를 먼저 확인합니다. 예제 3-1 (a) 조립 셀이 파트 1 · 2 의 부품을 다시 만들므로 여기서 시작해도 됩니다.

같은 질문이라도 답 문장은 실행할 때마다 조금씩 다를 수 있습니다. 검색 결과(점수 · 문서 · 절)는 매번 같습니다.
예제 3-3 (f) 부터는 색인이 바뀌므로 셀을 건너뛰지 않고 순서대로 실행합니다. 건너뛰면 뒤의 숫자가 달라집니다.

## 예제 3-1 · 절 발췌를 붙여 출처 있는 답 받기

**목적** 2일차 예제 3-3 (8) 에서 문서 한 건을 통째로 붙였던 자리를 절 발췌와 문서번호로 바꿉니다. 같은 질문(ESC 온도 상한과 조치)을 문서 없이 한 번(2일차 예제 3-1 (1) 과 같은 대조군), 검색으로 고른 절 발췌 4건을 붙여 한 번 Claude 에 보내
답에 구체적인 수치와 출처가 생기는지, 붙이는 건수(k)를 바꾸면 답이 어떻게 달라지는지 확인합니다.

### (a) 조립 셀: 부품을 한 셀로 모으기

파트 1 · 2 에서 따로 만든 부품을 한 셀에 모아 RAG 를 한 번에 준비합니다. 위에서부터 이렇게 이어집니다.

- **청킹 · 꼬리표**: `corpus/` 문서를 `## 절` 단위로 자르고, 문서 목록 파일에서 문서번호 · 버전 · 유형 · 날짜 꼬리표를 붙여 `chunks` 에 담습니다.
- **인덱스**: 청크마다 임베딩해 행렬 `E` 를 만듭니다(파트 1 예제 1-1 과 같은 방식).
- **검색** `search(q, k)`: 질문과 가장 닮은 청크 k 개를 점수와 함께 돌려줍니다.
- **Claude 호출** `llm(prompt)`: 글 한 덩어리를 Claude 에 보내고 답 글자만 돌려줍니다. `client.messages.create(...)` 한 번이 API 호출 한 번이고, `model` 은 쓸 모델 이름, `max_tokens` 는 답 길이 상한, `messages` 는 보낼 대화입니다. 돌아온 `r.content` 는 답 조각 목록이라 글자 조각만 이어 붙입니다.
- **프롬프트 틀** `TEMPLATE`: 발췌(`{ctx}`)와 질문(`{q}`)을 끼워 넣을 틀입니다. 네 칸은 예제 3-2 에서 봅니다.
- **RAG 한 번에** `ask(q)`: 검색 → 발췌를 `TEMPLATE` 에 채움 → `llm` 호출을 차례로 하고, 답과 검색 결과를 함께 돌려줍니다.

**볼 것** 진행 막대 뒤에 「청크 757 · 인덱스 (757, 384) · 모델 claude-sonnet-5-5」가 찍히면 준비가 끝난 것입니다. 이 셀은 아직 API 를 호출하지 않으므로 요금이 들지 않습니다.

In [ ]:
# 3-1 (a) 조립 셀: 파트 1 · 2 에서 만든 부품(청킹 · 꼬리표 · 인덱스 · 검색)과 답변 생성을 한 셀로 모읍니다
import os, re, csv, unicodedata
import numpy as np
from collections import namedtuple
from sentence_transformers import SentenceTransformer
import anthropic

CORPUS = 'corpus'                      # 실습 미션 3 에서는 'corpus_m3' 로 바꿉니다
MANIFEST = 'corpus_manifest.csv'       # 실습 미션 3 에서는 'corpus_m3_manifest.csv' 로 바꿉니다
model = SentenceTransformer('intfloat/multilingual-e5-small')
META = {unicodedata.normalize('NFC', r['file']): r
        for r in csv.DictReader(open(MANIFEST, encoding='utf-8-sig'))}

Chunk = namedtuple('Chunk', 'doc_id version type date section text')
chunks = []
for raw in sorted(os.listdir(CORPUS)):
    fn = unicodedata.normalize('NFC', raw)         # 지우면 한글 파일명에서 KeyError
    if not fn.endswith('.md'):
        continue
    m = META[fn]
    head, *secs = re.split(r'\n(?=## )', open(f'{CORPUS}/{raw}', encoding='utf-8').read())
    for s in [head] + secs:
        s = s.strip()
        if not s:                  # 머리 조각이 빈 문서는 건너뜀
            continue
        if secs and s == head.strip():
            s = '\n'.join(s.splitlines()[:8])          # 머리 청크는 제목·메타까지만
        chunks.append(Chunk(m['doc_id'], m['version'], m['type'], m['date'],
                            s.splitlines()[0].lstrip('# ').strip(), s[:1200]))

E = model.encode(['passage: ' + c.text for c in chunks],
                 normalize_embeddings=True, batch_size=64)

Hit = namedtuple('Hit', 'score doc_id version type date section text')

def search(q, k=4, exclude_types=()):
    qe = model.encode(['query: ' + q], normalize_embeddings=True)[0]
    cand = [i for i, c in enumerate(chunks) if c.type not in exclude_types]
    scores = E[cand] @ qe
    return [Hit(round(float(scores[j]), 3), *chunks[cand[j]])
            for j in scores.argsort()[::-1][:k]]

def format_ctx(hits):
    return '\n\n'.join(f'[{h.doc_id}_{h.version}] ({h.section})\n{h.text}'
                        for h in hits)

client = anthropic.Anthropic()          # ANTHROPIC_API_KEY 환경변수를 읽습니다
MODEL = 'claude-sonnet-5-5'

def llm(prompt, max_tokens=1500):                 # 글 한 덩어리를 보내고 답 글자만 돌려받습니다
    r = client.messages.create(                    # API 호출 한 번 = 요청 한 번(쓴 토큰만큼 요금)
        model=MODEL, max_tokens=max_tokens,        # 쓸 모델 · 답 길이 상한(토큰)
        output_config={'effort': 'low'},            # 짧은 질의응답이라 추론을 적게 씁니다
        messages=[{'role': 'user', 'content': prompt}])   # 보낼 대화: 사용자 메시지 하나
    if r.stop_reason == 'refusal':                 # 왜 멈췄는지: 거절이면 안내 문장으로
        return '[안내] 모델이 답변을 거절했습니다.'
    return ''.join(b.text for b in r.content if b.type == 'text').strip()   # 답 조각 중 글자 조각만 이어 붙입니다

TEMPLATE = """너는 SL-FAB 공정기술 담당자다.
아래 사내 문서 발췌만 근거로 답한다.

[발췌]
{ctx}

규칙: 문장마다 근거 [문서번호]를 붙인다.
발췌에 없으면 '사내 문서에서 근거를 찾지 못했습니다' 라고만 답한다.

질문: {q}"""

def ask(q, k=4, exclude_types=(), template=None):
    hits = search(q, k, exclude_types)
    ctx = format_ctx(hits)
    return llm((template or TEMPLATE).format(ctx=ctx, q=q)), hits

print('청크', len(chunks), '· 인덱스', E.shape, '· 모델', MODEL)

# Claude Code 에 물어보기: 머리 청크를 8줄로 자르는 이유를 알려 주고 그 청크 하나를 찍어 줘

### (b) 대조군: 문서를 붙이지 않고 물었을 때

2일차 예제 3-1 (1) 과 같은 대조군입니다. 같은 질문을 발췌 없이 Claude 에 보냅니다. 이 노트북에서 `llm` 을 처음 호출하는 셀이라 실제 API 호출이 일어납니다. 답이 오기까지 몇 초 걸리고, 쓴 토큰만큼 요금이 붙습니다. `TypeError: Could not resolve authentication method` 가 나면 키가 환경 변수에 없는 것이니 환경 준비 (2) 셀부터 다시 실행합니다. `AuthenticationError`(401)가 나면 보안 비밀에 넣은 키 값을, `RateLimitError`(429)가 나면 잠시 뒤 다시 실행합니다.

**볼 것** SL-FAB 문서의 기준 수치 없이 일반론과 확인할 곳만 돌아오는지 확인합니다. 일반적인 참고 범위는 나올 수 있습니다.

In [ ]:
# 3-1 (b) 대조군: 문서를 붙이지 않고 물었을 때
Q = "에치 장비 ESC 온도 상한이 몇 도이고, 초과하면 어떻게 조치해야 하나요?"
print(llm(f"SL-FAB 반도체 팹 기준으로 답해 주세요. {Q}"))

# Claude Code 에 물어보기: 같은 질문을 SMALL_MODEL 로도 보내 답 길이를 비교해 줘

### (c) 검색 결과를 붙여 다시 묻기

앞에서 만든 인덱스에서 top-4 를 가져와, 발췌와 프롬프트를 직접 이어 붙입니다. 이 셀은 버전 없이 문서번호만 붙이고, 예제 3-1 (d) 부터는 조립 셀의 `ask` 를 씁니다.

**볼 것** 2일차의 문서 한 건 대신 절 네 개를 문서번호와 함께 붙입니다. 점수 네 줄에 어떤 절이 뽑혔는지, 답에 62.0°C 같은 문서 수치와 [SOP-ETCH-007] 출처가 붙었는지 확인합니다.

In [ ]:
# 3-1 (c) 검색 결과를 붙여 다시 묻기
hits = search(Q, k=4)
ctx = '\n\n'.join(f'[{h.doc_id}] {h.text}' for h in hits)
prompt = f"""다음 사내 문서 발췌만 근거로 답해줘.
근거로 쓴 문서번호를 [ ] 로 함께 적어줘.
{ctx}
질문: {Q}"""

for h in hits:                      # 붙는 발췌를 눈으로 먼저
    print(f'{h.score:.3f}  {h.doc_id}_{h.version}  ({h.section[:22]})')
print('---')
print(llm(prompt))

# Claude Code 에 물어보기: ctx 의 글자 수와 대략 토큰 수를 찍어 줘

### (d) k 를 늘려 보기

붙이는 발췌 수 k 를 1 · 4 · 12 로 바꿔 같은 질문을 보냅니다. 많이 붙일수록 좋아지는지 봅니다.

**볼 것** k=1 답에 상한 수치 대신 「근거를 찾지 못했습니다」가 있는지, k=4 · k=12 답에 62.0°C 가 있는지 확인합니다. k=1 은 상한값이 있는 「관리 기준」 절이 빠진 경우입니다. 발췌를 세 배로 붙인 k=12 의 답이 k=4 보다 나아졌는지 답해 봅니다.

In [ ]:
# 3-1 (d) k 를 1 · 4 · 12 로 바꿔 봅니다
for k in [1, 4, 12]:
    ans, hits = ask(Q, k=k)
    print(f'k={k:<2d} 답 {len(ans):4d}자 |', ans[:70].replace('\n', ' '))

# Claude Code 에 물어보기: k=1 일 때 뽑힌 절이 무엇인지 hits 를 같이 찍어 줘

## 예제 3-2 · RAG 프롬프트 네 칸 구성

**목적** 예제 3-1 (a) 조립 셀의 TEMPLATE 네 칸(역할 · 발췌 · 출력 규칙 · 근거 없음 규칙) 가운데 (4) 근거 없음 규칙을 빼고 없는 호기(ETCH-07)를 물어,
규칙이 있을 때와 없을 때 답이 어떻게 달라지는지 확인합니다.

### (a) (4) 규칙이 있을 때 없는 호기 묻기

없는 호기 ETCH-07 을 TEMPLATE 그대로 묻습니다.

**볼 것** 답이 '사내 문서에서 근거를 찾지 못했습니다' 로 시작합니다. 같은 질문을 예제 3-2 (b) 에서 (4) 규칙 없이 다시 보냅니다.

In [ ]:
# 3-2 (a) 없는 호기를 물으면 ((4) 규칙 있음)
ans, hits = ask('ETCH-07 호기 ESC 온도 기준은?')   # 존재하지 않는 호기
print(ans)

# Claude Code 에 물어보기: hits 점수를 찍어 근거 없음 답과 점수가 무관한지 확인해 줘

### (b) (4) 규칙을 빼고 같은 질문

TEMPLATE 에서 근거 없음 규칙 한 줄을 지웁니다.

**볼 것** 옆 호기 기준 58.0 ~ 62.0°C 가 「참고」로 답에 실립니다. 예제 3-2 (a) 의 답과 비교해, (4) 규칙의 「라고만 답한다」가 막던 내용이 무엇인지 답해 봅니다.

In [ ]:
# 3-2 (b) (4) 규칙을 빼고 같은 질문
TEMPLATE_NO4 = TEMPLATE.replace(
    "발췌에 없으면 '사내 문서에서 근거를 찾지 못했습니다' 라고만 답한다.", '')
ans, hits = ask('ETCH-07 호기 ESC 온도 기준은?', template=TEMPLATE_NO4)
print(ans)

# Claude Code 에 물어보기: TEMPLATE_NO4 에서 그 줄이 실제로 지워졌는지 확인하는 코드 짜 줘

## 예제 3-3 · RAG 가 깨지는 경우 보기

**목적** 실패 세 가지(관련 없는 문서가 상위를 차지 · 같은 문서의 구판과 개정판 충돌 · 절이 문서 맥락을 잃음)를 일부러 만들고,
각각 검색 결과의 어디에서 원인이 보이는지와 어떤 조치(질문 좁히기 · 메타 필터 · 구판 제외 · 제목 접두 색인)가 듣는지 확인합니다.

### (a) 실패 1 관련 없는 문서가 상위를 차지

회의록이 많은 코퍼스에서 이렇게 됩니다.

**볼 것** 검색 결과에 무엇이 올라왔는지부터 보세요.

In [ ]:
# 3-3 (a) 실패 1 관련 없는 문서가 상위를 차지
Q_NOISE = '최근 에치 장비들에서 있었던 센서 관련 품질 이슈를 정리해 주세요.'
ans, hits = ask(Q_NOISE, k=6)
for h in hits:
    print(f'{h.score:.3f}  {h.doc_id}_{h.version}  ({h.section[:20]})')
print(ans)

# Claude Code 에 물어보기: Q_NOISE 에서 '센서' 를 빼고 search 만 돌려 1위가 바뀌는지 봐 줘

### (b) 질문을 좁혀 다시

넓은 말 대신 장비 호기와 현상으로 묻습니다.

**볼 것** 회의록 대신 QIR 「현상」 절이 0.92 로 1 · 2위에 오고, 1위와 4위의 점수 차가 예제 3-3 (a) 보다 벌어졌는지 확인합니다. 답에 원인 · 조치가 빠진 이유는 예제 3-3 (g) 에서 다시 봅니다.

In [ ]:
# 3-3 (b) 질문을 좁혀 다시
ans, hits = ask('ETCH-03 호기 파티클 급증의 원인과 조치를 알려주세요')
for h in hits:
    print(f'{h.score:.3f}  {h.doc_id}_{h.version}  ({h.section[:20]})')
print(ans)

# Claude Code 에 물어보기: 1위와 4위의 점수 차를 (a) 와 나란히 찍어 줘

### (c) 메타데이터 필터를 걸어 다시

(심화 · 수업에서 넘어갈 수 있음) 넘어가도 뒤 셀은 그대로 돌아갑니다.

회의록을 빼고 같은 질문을 던집니다.

**볼 것** 회의록은 빠졌지만 점수가 0.857~0.855 로 거의 같습니다. 답은 '근거 없음' 그대로입니다.

In [ ]:
# 3-3 (c) 메타데이터 필터를 걸어 다시 (회의록 제외)
ans, hits = ask(Q_NOISE, k=6, exclude_types=['회의록'])
for h in hits:
    print(f'{h.score:.3f}  {h.doc_id}_{h.version}  ({h.section[:20]})')
print(ans)

# Claude Code 에 물어보기: exclude 뒤 후보 수(cand 길이)를 search 안에서 찍어 줘

### (d-) 두 개정본은 얼마나 닮았나 (보조)

구판과 개정판의 같은 절을 글자와 벡터로 비교합니다.

**볼 것** 두 절 사이 유사도 0.999. 검색으로는 구판을 가려낼 수 없습니다.

**짚을 줄**
- `difflib.SequenceMatcher(...).get_matching_blocks()` 는 두 글에서 같은 구간들을 찾습니다. 그 길이를 더한 값이 공통 글자 수입니다.
- `va @ vb` 는 두 절 벡터의 유사도입니다. 1 에 가까울수록 검색에서는 같은 글로 보입니다.

In [ ]:
# 3-3 (d-) 두 개정본은 얼마나 닮았나 (보조)
import difflib

pair = {c.version: c for c in chunks
        if c.doc_id == 'SOP-ETCH-012' and c.section == '설정 기준'}
a, b = pair['v1.2'].text, pair['v2.0'].text
common = sum(bl.size for bl in difflib.SequenceMatcher(None, a, b).get_matching_blocks())
print(f'공통 {common}자   v1.2 {len(a)}자 · v2.0 {len(b)}자')

Q_HE = '에치 웨이퍼 클램핑 시 He 배면압 기준은?'
qe = model.encode(['query: ' + Q_HE], normalize_embeddings=True)[0]
va, vb = model.encode([f'passage: {a}', f'passage: {b}'], normalize_embeddings=True)
print(f'질문 대비   v1.2 {va @ qe:.3f} · v2.0 {vb @ qe:.3f}')
print(f'두 절 사이  {va @ vb:.3f}   ← 검색으로는 구판을 가려낼 수 없습니다')

# Claude Code 에 물어보기: 두 절에서 서로 다른 글자를 difflib 로 찍어 줘

### (d) 실패 2 버전 충돌

같은 문서의 구판·개정판이 둘 다 있습니다.

**볼 것** v2.0 과 v1.2 의 같은 절이 0.926 동점입니다. 검색은 구판을 가려내지 못합니다.

질문 `Q_HE` 는 바로 위 예제 3-3 (d-) 셀에서 만듭니다. 예제 3-3 (d-) 를 건너뛰었다면 그 셀부터 실행합니다.

In [ ]:
# 3-3 (d) 실패 2 버전 충돌
ans, hits = ask(Q_HE)
for h in hits:
    print(f'{h.score:.3f}  {h.doc_id}_{h.version}  ({h.section[:20]})')
print(ans)

# Claude Code 에 물어보기: 같은 doc_id 면 최신 version 만 남기는 후처리 함수 짜 줘

### (e) 버전 표기를 지우면

(심화 · 수업에서 넘어갈 수 있음) 넘어가도 뒤 셀은 그대로 돌아갑니다.

발췌의 [문서번호_버전] 에서 버전을 지우고 다시 묻습니다.

**볼 것** 8.0 과 6.0 Torr 를 둘 다 적고 현행 값을 단정하지 못합니다. 답 모양은 실행마다 달라, 판과 값을 거꾸로 짝짓는 답이 나오기도 합니다.

In [ ]:
# 3-3 (e) 버전 표기를 지우면
hits = search(Q_HE)
ctx = '\n\n'.join(f'[{h.doc_id}] ({h.section})\n{h.text}' for h in hits)
print(llm(TEMPLATE.format(ctx=ctx, q=Q_HE)))

# Claude Code 에 물어보기: ctx 에 version 대신 date 만 붙이면 답이 달라지는지 시험해 줘

### (f) 구판을 색인에서 제외

SOP-ETCH-012 v1.2 를 색인에서 빼면 757 → 754 청크가 됩니다.

**볼 것** 개정판 8.0 Torr ± 0.4 만 답에 남습니다.

**짚을 줄**
- `keep` 은 SOP-ETCH-012 v1.2 가 아닌 청크의 번호 목록입니다. `E[keep]` 처럼 번호 목록으로 행을 고르면 청크 목록과 벡터가 같은 순서로 함께 줄어듭니다.

In [ ]:
# 3-3 (f) 구판을 색인에서 제외
# ⚠ 이 셀부터 색인이 757 → 754 청크로 바뀝니다. 뒤의 (g) · (h) · 3-4 는 이 색인에서 돌아갑니다.
keep = [i for i, c in enumerate(chunks)
        if not (c.doc_id == 'SOP-ETCH-012' and c.version == 'v1.2')]
chunks, E = [chunks[i] for i in keep], E[keep]
ans, hits = ask(Q_HE)
for h in hits:
    print(f'{h.score:.3f}  {h.doc_id}_{h.version}')
print(ans)

# Claude Code 에 물어보기: keep 으로 지운 청크 3개의 절 이름을 찍어 줘

### (g) 실패 3 절이 문서 맥락을 잃기

원인도 조치도 한 절에 다 있는데 그 절이 검색에 뽑히지 않는 경우입니다.

**볼 것** 원인과 조치는 QIR-2026-0712 「시정 조치」 절에 함께 있는데, top-4 에 그 절이 없습니다. 답에서 원인이 빠졌는지 확인합니다. 그 절이 몇 위인지는 예제 3-3 (g-) 에서 잽니다.

In [ ]:
# 3-3 (g) 실패 3 절이 문서 맥락을 잃기
Q_D = 'ETCH-04 호기 ESC 온도 드리프트 원인과 재발 방지 조치는?'
ans, hits = ask(Q_D)
for h in hits:
    print(f'{h.score:.3f}  {h.doc_id}_{h.version}  ({h.section[:20]})')
print(ans)

# Claude Code 에 물어보기: '시정 조치' 절 본문이 질문과 어떤 낱말을 공유하는지 보여 줘

### (g-) 원인 절은 지금 몇 위인가 (보조)

제목 접두 색인 전후로 「시정 조치」 절의 순위를 잽니다. 754 청크를 다시 임베딩하므로 다른 셀보다 오래 걸립니다.

**볼 것** 292위 / 754 에서 4위 / 754 로 올라갑니다.

**짚을 줄**
- `TITLE` 은 (문서번호, 버전)마다 첫 청크의 절 이름, 곧 문서 제목을 담습니다. `setdefault` 는 처음 본 값만 넣습니다.
- `rank_of_section` 은 색인(`E` 또는 `E_title`)에서 지정한 문서의 지정한 절이 몇 위인지와 그 점수를 돌려줍니다. 예제 1-1 의 `rank_of`(문서 한 건의 순위)와는 다른 함수입니다.
- `E_title` 은 본문 앞에 「문서 제목 · 절 이름」을 붙여 다시 임베딩한 색인입니다.

In [ ]:
# 3-3 (g-) 원인 절은 지금 몇 위인가 (보조)
TITLE = {}
for c in chunks:
    TITLE.setdefault((c.doc_id, c.version), c.section)    # 첫 청크의 절 제목 = 문서 제목

def rank_of_section(index, doc_id, section):
    qe = model.encode(['query: ' + Q_D], normalize_embeddings=True)[0]
    sc = index @ qe
    for r, i in enumerate(sc.argsort()[::-1], 1):
        if chunks[i].doc_id == doc_id and chunks[i].section == section:
            return r, float(sc[i])

E_title = model.encode([f'passage: {TITLE[c.doc_id, c.version]} · {c.section}\n{c.text}'
                        for c in chunks], normalize_embeddings=True, batch_size=64)
if rank_of_section(E, 'QIR-2026-0712', '시정 조치') is None:     # 미션 3 코퍼스(corpus_m3)에는 이 문서가 없습니다
    print('이 묶음에는 QIR-2026-0712 「시정 조치」 절이 없어 순위 비교는 건너뜁니다 · 제목 접두 색인', E_title.shape)
else:
    r, s = rank_of_section(E, 'QIR-2026-0712', '시정 조치')
    print(f'제목 접두 전   {r}위 / {len(chunks)}   ({s:.3f})')
    r, s = rank_of_section(E_title, 'QIR-2026-0712', '시정 조치')
    print(f'제목 접두 후   {r}위 / {len(chunks)}   ({s:.3f})   ← 원인·조치가 함께 있는 절')

# Claude Code 에 물어보기: 접두에 절 이름 없이 문서 제목만 붙이면 순위가 어떻게 되는지 재 줘

### (h) 제목 접두 색인으로 다시

절 앞에 문서 제목을 붙여 만든 색인(`E_title`, 예제 3-3 (g-) 에서 만듦)으로 같은 질문을 던집니다. 이 셀부터 색인 `E` 가 제목 접두 색인으로 바뀌고, 예제 3-4 도 이 색인에서 돌아갑니다.

**볼 것** 원인(칠러 배관 부분 폐색)과 조치가 함께 답에 들어옵니다.

In [ ]:
# 3-3 (h) 제목 접두 색인으로 다시
assert len(E_title) == len(chunks), '3-3 (g-) 셀을 먼저 실행하세요 (조립 셀을 다시 돌렸으면 (g-) 도 다시)'
E = E_title                                   # 절 앞에 문서 제목을 붙여 만든 색인으로 교체
ans, hits = ask(Q_D)
for h in hits:
    print(f'{h.score:.3f}  {h.doc_id}_{h.version}  ({h.section[:20]})')
print(ans)

# Claude Code 에 물어보기: Q 로 search 해 접두 전 점수와 나란히 비교해 줘

## 예제 3-4 · 답을 믿어도 되는지 확인하는 절차 · 첫 질의응답 테스트

**목적** 답과 함께 검색 결과(점수 · 문서 · 버전 · 절)를 한 화면에 찍는 진단 함수를 만들고, 성격이 다른 질문 여섯 개로
첫 질의응답 테스트를 돌려 정상 답과 실패 사례를 표 하나에 기록합니다.

### (a) 디버깅용 출력

(심화 · 수업에서 넘어갈 수 있음) 넘어가도 뒤 셀은 그대로 돌아갑니다.

답과 함께 질문 · 점수 · 문서번호 · 버전 · 본문 앞부분을 항상 찍어 둡니다.

**볼 것** 네 줄의 점수 · 문서번호 · 버전 · 본문 앞부분을 답의 [출처] 와 대조합니다.

In [ ]:
# 3-4 (a) 디버깅용 출력
ans, hits = ask(Q)
print('질문:', Q)
for h in hits:
    print(f'  {h.score:.3f} [{h.doc_id}_{h.version}] {h.text[:40]}')
print('답:', ans)

# Claude Code 에 물어보기: 답에 나온 [문서번호] 가 hits 에 모두 있는지 검사하는 코드 짜 줘

### (b) 검색 결과를 답과 함께 보여주기

(심화 · 수업에서 넘어갈 수 있음) 넘어가도 뒤 셀은 그대로 돌아갑니다.

답 아래에 근거 절을 함께 찍습니다.

**볼 것** 답의 각 문장이 「--- 근거 ---」 아래 어느 절에서 왔는지 짝지어 보고, 짝이 없는 문장이 있는지 확인합니다.

In [ ]:
# 3-4 (b) 답과 근거를 한 화면에
ans, hits = ask(Q)
print(ans)
print('--- 근거 ---')
for h in hits:
    print(f'[{h.doc_id} {h.version}] {h.section}: {h.text[:60]}')

# Claude Code 에 물어보기: 답 문장마다 근거 절 이름을 자동으로 짝짓는 코드 짜 줘

### (c) 진단 셀 하나로 묶기

답이 이상할 때마다 눈으로 확인하던 것을 함수 하나로 만듭니다.

**볼 것** 출력 사전의 「1위-4위 차」 · 「유형」 · 「같은 문서 여러 판」 칸을 확인합니다. 점수만으로는 실패를 가르지 못하므로, 유형이 한쪽으로 쏠렸는지와 같은 문서의 여러 판이 왔는지를 같이 봅니다.

**짚을 줄**
- `diagnose` 는 `ask` 로 답과 검색 결과를 받은 뒤, 눈으로 보던 항목을 한 줄(`row`)로 정리합니다.
- `len(vers) > len(docs_)` 는 문서번호보다 (문서번호, 버전) 쌍이 많을 때, 곧 같은 문서의 판이 둘 이상 섞였을 때 참입니다.
- `re.search(r'\[[A-Z]+-', ans)` 는 답에 「[SOP-」 같은 대괄호 출처가 있는지 봅니다.

In [ ]:
# 3-4 (c) 진단 셀 하나로 묶기
NO_EVIDENCE = '근거를 찾지 못했습니다'

def diagnose(q, k=4):
    ans, hits = ask(q, k=k)
    docs_ = {h.doc_id for h in hits}
    vers = {(h.doc_id, h.version) for h in hits}
    row = {'질문': q[:26],
           '최고 점수': hits[0].score,
           '1위-4위 차': round(hits[0].score - hits[min(3, len(hits) - 1)].score, 3),
           '유형': '/'.join(h.type for h in hits),
           '같은 문서 여러 판': '있음' if len(vers) > len(docs_) else '없음',
           '출처 표기': '있음' if re.search(r'\[[A-Z]+-', ans) else '없음',
           '근거 없음': '예' if NO_EVIDENCE in ans else '아니오'}
    return row, ans

row, _ = diagnose(Q)
print(row)

# Claude Code 에 물어보기: row 에 서로 다른 문서 수 칸을 추가해 줘

### (d) 첫 질의응답 테스트

성격이 다른 질문 여섯 개를 표 하나로 봅니다.

**볼 것** 점수만 보면 없는 호기(0.914)가 정상(0.840)보다 높습니다. 점수 차와 근거 없음 칸을 같이 봅니다. 「출처 표기」 · 「근거 없음」 칸은 Claude 답에서 세므로 실행마다 달라질 수 있습니다. 답은 `answers` 에 질문 순서대로 들어 있으니, 새 셀에서 `print(answers[2])` 처럼 열어 근거와 대조합니다.

**짚을 줄**
- `Q` 는 예제 3-1 (b), `Q_NOISE` 는 예제 3-3 (a) 에서 만든 질문입니다. 노이즈 질문은 예제 3-3 (a) 와 같은 k=6 으로 돌립니다. 질문마다 Claude API 를 한 번씩, 모두 여섯 번 호출합니다.
- `TESTS` 는 (예상 유형, 질문, k) 여섯 줄입니다. 반복문이 질문마다 `diagnose` 를 부르고, `{'예상': kind, **row}` 로 예상 유형 칸을 맨 앞에 붙입니다.
- `pd.DataFrame(rows)` 로 줄들을 표로 만들어 한 번에 찍습니다.

In [ ]:
# 3-4 (d) 첫 질의응답 테스트: 성격이 다른 질문 여섯 개를 돌려 표 하나로 봅니다
import pandas as pd
TESTS = [('정상',      Q, 4),
         ('정상',      '시즈닝 런은 더미 웨이퍼 몇 장으로 하나요?', 4),
         ('노이즈',    Q_NOISE, 6),
         ('버전',      '에치 웨이퍼 클램핑 시 He 배면압 허용 편차는?', 4),
         ('없는 호기', 'ETCH-07 호기 ESC 온도 기준은?', 4),
         ('문서 밖',   '야간조 식당 메뉴가 뭐예요?', 4)]
rows, answers = [], []
for kind, q, k in TESTS:
    row, ans = diagnose(q, k)
    rows.append({'예상': kind, **row}); answers.append(ans)
pd.set_option('display.width', 200)
print(pd.DataFrame(rows).to_string(index=False))

# Claude Code 에 물어보기: 표를 CSV 로, answers 를 텍스트로 함께 저장하는 코드 짜 줘